# AlpCAN nnU-Net: Gelismis Nodul Segmentasyonu

**Amac:** nnU-Net v2 framework'u ile LIDC-IDRI uzerinde nodul segmentasyonu.
Mevcut U-Net (NB06, Dice 0.622) baseline'ini iyilestirmek.

**Hedef:** Dice > 0.75

**Model:** nnU-Net v2 (2D konfigurasyon)
**Dataset:** zhangweiled/lidcidri (LIDC-IDRI slices)
**GPU:** Kaggle T4 16GB
**Egitim:** 250 epoch, fold 0

In [ ]:
!pip install -q nnunetv2

In [ ]:
import os
import json
import time
import numpy as np
import shutil
from pathlib import Path
from PIL import Image
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# === P100/eski GPU icin Triton ve torch.compile devre disi ===
# nnU-Net v2 PyTorch torch.compile kullanir, Triton backend
# compute capability >= 7.0 gerektirir. P100 (6.0) desteklenmiyor.
os.environ['PYTORCH_JIT'] = '0'  # JIT compiler devre disi
os.environ['TORCHINDUCTOR_DISABLE'] = '1'  # Inductor backend devre disi

# nnU-Net ortam degiskenleri -- nnunetv2 import'undan ONCE ayarlanmali
os.environ['nnUNet_raw'] = '/kaggle/working/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/kaggle/working/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/kaggle/working/nnUNet_results'

for d in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
    os.makedirs(d, exist_ok=True)

import torch

# torch.compile'i devre disi birak (P100 Triton desteklemiyor)
if hasattr(torch, '_dynamo'):
    torch._dynamo.config.suppress_errors = True
    torch._dynamo.config.disable = True

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Bellek: {mem_gb:.1f} GB")
    print(f"Compute Capability: {cap[0]}.{cap[1]}")
    if cap[0] < 7:
        print("UYARI: GPU Triton desteklemiyor (< 7.0), torch.compile devre disi.")

In [ ]:
# === Konfigurasyon ===
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
DATASET_ID = 501
DATASET_NAME = f"Dataset{DATASET_ID:03d}_LIDC"

# Veri seti yolu
DATA_DIR = Path("/kaggle/input/lidcidri/LIDC-IDRI-slices")

# Alternatif yol arama
if not DATA_DIR.exists():
    for candidate in INPUT_DIR.rglob("LIDC-IDRI-0001"):
        DATA_DIR = candidate.parent
        break

print(f"Veri dizini: {DATA_DIR}")
print(f"Veri dizini mevcut: {DATA_DIR.exists()}")

# Hasta sayisi kontrol -- sadece LIDC-IDRI-XXXX formatindaki dizinleri al
if DATA_DIR.exists():
    patient_dirs = sorted([
        d for d in DATA_DIR.iterdir()
        if d.is_dir() and d.name.startswith("LIDC-IDRI-")
    ])
    print(f"Toplam hasta: {len(patient_dirs)}")
    if patient_dirs:
        print(f"  Ilk: {patient_dirs[0].name}")
        print(f"  Son: {patient_dirs[-1].name}")
else:
    print("HATA: Veri dizini bulunamadi!")
    patient_dirs = []

---
## 2. Veri Donusumu: PNG -> nnU-Net Formati

nnU-Net v2 PNG dosyalarini dogrudan destekler. Veriler asagidaki formata donusturulur:

```
nnUNet_raw/Dataset501_LIDC/
  imagesTr/LIDC_00000_0000.png   (goruntu, _0000 = kanal 0)
  labelsTr/LIDC_00000.png        (maske, kanal yok)
  dataset.json
```

In [ ]:
# === PNG -> nnU-Net format donusumu ===
import pandas as pd

NNUNET_RAW = Path(os.environ['nnUNet_raw'])
DATASET_DIR = NNUNET_RAW / DATASET_NAME
IMAGES_TR = DATASET_DIR / "imagesTr"
LABELS_TR = DATASET_DIR / "labelsTr"

# Temiz baslangic
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
IMAGES_TR.mkdir(parents=True, exist_ok=True)
LABELS_TR.mkdir(parents=True, exist_ok=True)

print(f"Hedef dizin: {DATASET_DIR}")
print(f"Donusum basliyor...")
print("=" * 60)

start_time = time.time()
sample_idx = 0
skipped_empty = 0
skipped_error = 0
positive_masks = 0
total_patients_processed = 0
all_widths = []
all_heights = []

# Istatistik toplama
patient_sample_counts = defaultdict(int)
annotator_counts = []

for p_idx, pdir in enumerate(patient_dirs):
    nodule_dirs = sorted([d for d in pdir.iterdir() if d.is_dir()])
    patient_has_samples = False

    for ndir in nodule_dirs:
        img_dir = ndir / "images"
        mask_dirs = sorted(
            [d for d in ndir.iterdir() if d.is_dir() and d.name.startswith("mask")]
        )

        # Goruntu veya maske dizini yoksa atla
        if not img_dir.exists() or not mask_dirs:
            continue

        for img_file in sorted(img_dir.glob("*.png")):
            try:
                # Goruntu oku
                img = Image.open(img_file).convert('L')
                img_np = np.array(img, dtype=np.uint8)
                h, w = img_np.shape[:2]

                # Boyut kontrolu -- cok kucuk goruntuler atla
                if h < 8 or w < 8:
                    skipped_error += 1
                    continue

                # Konsensus maskesi olustur -- tum annotatorlerden
                mask_sum = np.zeros((h, w), dtype=np.float32)
                n_masks = 0

                for mdir in mask_dirs:
                    mask_file = mdir / img_file.name
                    if not mask_file.exists():
                        continue
                    try:
                        m = Image.open(mask_file).convert('L')
                        m_np = np.array(m, dtype=np.uint8)

                        # Boyut uyumsuzlugu -- yeniden boyutlandir
                        if m_np.shape[:2] != (h, w):
                            m = m.resize((w, h), Image.NEAREST)
                            m_np = np.array(m, dtype=np.uint8)

                        mask_sum += (m_np > 0).astype(np.float32)
                        n_masks += 1
                    except Exception:
                        continue

                if n_masks == 0:
                    skipped_empty += 1
                    continue

                # Konsensus: annotatorlerin >= %50'si isaretlemis
                consensus = (mask_sum / n_masks >= 0.5).astype(np.uint8)
                annotator_counts.append(n_masks)

                # nnU-Net isimlendirme: LIDC_XXXXX_0000.png (goruntu), LIDC_XXXXX.png (etiket)
                case_id = f"LIDC_{sample_idx:05d}"

                # Goruntu kaydet -- grayscale PNG (8-bit)
                img_out = Image.fromarray(img_np, mode='L')
                img_out.save(IMAGES_TR / f"{case_id}_0000.png")

                # Etiket kaydet -- 0=arkaplan, 1=nodul
                label_out = Image.fromarray(consensus, mode='L')
                label_out.save(LABELS_TR / f"{case_id}.png")

                # Istatistik
                all_widths.append(w)
                all_heights.append(h)
                if consensus.max() > 0:
                    positive_masks += 1

                patient_sample_counts[pdir.name] += 1
                patient_has_samples = True
                sample_idx += 1

            except Exception as e:
                skipped_error += 1
                continue

    if patient_has_samples:
        total_patients_processed += 1

    # Her 100 hastada ilerleme raporu
    if (p_idx + 1) % 100 == 0:
        elapsed = time.time() - start_time
        print(
            f"  [{p_idx + 1:>4d}/{len(patient_dirs)}] "
            f"{sample_idx:,} ornek | "
            f"{elapsed:.0f}s"
        )

elapsed = time.time() - start_time
print(f"\nDonusum tamamlandi! ({elapsed:.1f}s)")
print("=" * 60)
print(f"Toplam ornek:        {sample_idx:,}")
print(f"Islenen hasta:       {total_patients_processed}")
print(f"Pozitif maskeli:     {positive_masks:,} ({positive_masks / max(sample_idx, 1) * 100:.1f}%)")
print(f"Atlanan (bos):       {skipped_empty}")
print(f"Atlanan (hata):      {skipped_error}")
if all_widths:
    print(f"Goruntu boyutu:      {min(all_widths)}x{min(all_heights)} - {max(all_widths)}x{max(all_heights)}")
    print(f"Ortalama boyut:      {np.mean(all_widths):.0f}x{np.mean(all_heights):.0f}")
if annotator_counts:
    print(f"Annotator (ort):     {np.mean(annotator_counts):.1f}")

# === dataset.json olustur ===
# nnU-Net v2 icin gerekli format
dataset_json = {
    "channel_names": {
        "0": "CT"
    },
    "labels": {
        "background": 0,
        "nodule": 1
    },
    "numTraining": sample_idx,
    "file_ending": ".png"
}

dataset_json_path = DATASET_DIR / "dataset.json"
with open(dataset_json_path, 'w') as f:
    json.dump(dataset_json, f, indent=2)

print(f"\ndataset.json kaydedildi: {dataset_json_path}")
print(json.dumps(dataset_json, indent=2))

In [ ]:
# === Veri seti dogrulama ===
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')

# Dosya sayilarini kontrol et
img_files = sorted(IMAGES_TR.glob("*.png"))
lbl_files = sorted(LABELS_TR.glob("*.png"))
print(f"imagesTr: {len(img_files)} dosya")
print(f"labelsTr: {len(lbl_files)} dosya")

# Her goruntunun eslestirilmis bir etiketi var mi?
img_ids = {f.stem.replace('_0000', '') for f in img_files}
lbl_ids = {f.stem for f in lbl_files}

if img_ids != lbl_ids:
    diff = img_ids.symmetric_difference(lbl_ids)
    print(f"UYARI: {len(diff)} eslesmeyen dosya bulundu, temizleniyor...")
    print(f"  Fark (ilk 10): {list(diff)[:10]}")

    # Sadece LIDC_XXXXX formatindaki dosyalari tut, gerisi silinsin
    import re
    valid_pattern = re.compile(r'^LIDC_\d{5}$')

    # Gecersiz ID'li image dosyalarini sil
    for f in list(IMAGES_TR.glob("*.png")):
        cid = f.stem.replace('_0000', '')
        if not valid_pattern.match(cid):
            f.unlink(missing_ok=True)
            print(f"  Silindi (gecersiz image): {f.name}")

    # Gecersiz ID'li label dosyalarini sil
    for f in list(LABELS_TR.glob("*.png")):
        cid = f.stem
        if not valid_pattern.match(cid):
            f.unlink(missing_ok=True)
            print(f"  Silindi (gecersiz label): {f.name}")

    # Yeniden say
    img_files = sorted(IMAGES_TR.glob("*.png"))
    lbl_files = sorted(LABELS_TR.glob("*.png"))
    img_ids = {f.stem.replace('_0000', '') for f in img_files}
    lbl_ids = {f.stem for f in lbl_files}

    # Hala uyumsuzluk varsa, eslesmeyen taraftaki dosyalari sil
    only_in_img = img_ids - lbl_ids
    only_in_lbl = lbl_ids - img_ids
    for cid in only_in_img:
        (IMAGES_TR / f"{cid}_0000.png").unlink(missing_ok=True)
    for cid in only_in_lbl:
        (LABELS_TR / f"{cid}.png").unlink(missing_ok=True)

    # dataset.json guncelle
    img_files = sorted(IMAGES_TR.glob("*.png"))
    lbl_files = sorted(LABELS_TR.glob("*.png"))
    img_ids = {f.stem.replace('_0000', '') for f in img_files}
    lbl_ids = {f.stem for f in lbl_files}

    dataset_json["numTraining"] = len(img_files)
    with open(dataset_json_path, 'w') as f:
        json.dump(dataset_json, f, indent=2)
    print(f"  Temizlendi. imagesTr: {len(img_files)}, labelsTr: {len(lbl_files)}")

assert img_ids == lbl_ids, f"Goruntu-etiket uyumsuzlugu hala mevcut! Fark: {img_ids.symmetric_difference(lbl_ids)}"
print(f"Goruntu-etiket eslesmesi: OK ({len(img_ids)} cift)")

# Ornek gorsellestirme (8 ornek)
n_show = min(8, len(img_files))
fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))
if n_show == 1:
    axes = axes.reshape(2, 1)
fig.suptitle('nnU-Net Formati: Ornek Goruntuler ve Etiketler', fontsize=14)

# Pozitif maskeli ornekler bul (daha bilgilendirici gorsel icin)
positive_indices = []
for i, lf in enumerate(lbl_files):
    if i >= 500:  # Arama limitini koy
        break
    lbl_check = np.array(Image.open(lf).convert('L'))
    if lbl_check.max() > 0:
        positive_indices.append(i)
    if len(positive_indices) >= n_show:
        break

# Yeterli pozitif yoksa, mevcut ornekleri kullan
if len(positive_indices) < n_show:
    remaining = n_show - len(positive_indices)
    step = max(1, len(img_files) // remaining)
    for i in range(0, len(img_files), step):
        if i not in positive_indices:
            positive_indices.append(i)
        if len(positive_indices) >= n_show:
            break

show_indices = positive_indices[:n_show]

for col, idx in enumerate(show_indices):
    # Goruntu
    img_path = img_files[idx]
    case_id = img_path.stem.replace('_0000', '')
    lbl_path = LABELS_TR / f"{case_id}.png"

    img_np = np.array(Image.open(img_path).convert('L'))
    lbl_np = np.array(Image.open(lbl_path).convert('L'))

    axes[0, col].imshow(img_np, cmap='gray')
    axes[0, col].set_title(f'{case_id}', fontsize=8)
    axes[0, col].axis('off')

    axes[1, col].imshow(img_np, cmap='gray')
    if lbl_np.max() > 0:
        axes[1, col].contour(lbl_np, colors='lime', linewidths=1.5, levels=[0.5])
    axes[1, col].set_title('+ Maske', fontsize=8)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Goruntu', fontsize=11)
axes[1, 0].set_ylabel('Etiket', fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nnunet_dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()

# Goruntu boyutu dagilimi
print(f"\nDataset istatistikleri:")
print(f"  Goruntu sayisi: {len(img_files)}")
print(f"  Pozitif maske:  {positive_masks} ({positive_masks / max(len(img_files), 1) * 100:.1f}%)")
print(f"  dataset.json:   OK")

---
## 3. nnU-Net On-Isleme (Plan & Preprocess)

nnU-Net otomatik olarak:
- Goruntu boyutlarini analiz eder
- Normalizasyon parametrelerini hesaplar
- Patch size, batch size belirler
- 2D konfigurasyon icin veri setini hazirlar

In [ ]:
import subprocess, sys

# nnU-Net plan & preprocess -- timeout koruması ile
print("nnUNetv2_plan_and_preprocess başlatılıyor...")
print("Bu işlem 10-30 dakika sürebilir.\n")

try:
    result = subprocess.run(
        ["nnUNetv2_plan_and_preprocess", "-d", "501", "--verify_dataset_integrity", "-c", "2d"],
        capture_output=True, text=True, timeout=3600  # 1 saat timeout
    )
    print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
    if result.returncode != 0:
        print(f"\nSTDERR (son 2000 karakter):")
        print(result.stderr[-2000:])
        print(f"\nReturn code: {result.returncode}")
    else:
        print("\nPlan & Preprocess başarıyla tamamlandı!")
except subprocess.TimeoutExpired:
    print("HATA: Plan & preprocess 1 saat içinde tamamlanamadı!")
    sys.exit(1)
except Exception as e:
    print(f"HATA: {e}")
    sys.exit(1)

---
## 4. nnU-Net Egitim (50 Epoch, Fold 0)

- **Trainer:** `nnUNetTrainer__nnUNetPlans__2d` (varsayilan 1000 epoch trainer)
- **Epoch override:** `--npz` flag'i ile 50 epoch'a sinirlandirma (Kaggle 12-saat limiti icin)
- **Konfigurasyon:** 2D (dilim bazli segmentasyon)
- **Fold:** 0 (5-fold'un ilki, zaman kisitlamasi nedeniyle)
- nnU-Net otomatik augmentation, learning rate schedule ve early stopping uygular

In [ ]:
import subprocess, sys, signal

# nnU-Net egitim -- 50 epoch (Kaggle 12-saat limiti icin)
NUM_EPOCHS = 50

# nnU-Net v2 custom epoch: environment variable ile override
os.environ['nnUNet_n_proc_DA'] = '2'  # Data augmentation paralel isci sayisi

print(f"nnUNetv2_train baslatiliyor... (Dataset 501, 2d, fold 0, {NUM_EPOCHS} epoch)")
print(f"Tahmini sure: 3-6 saat (T4 GPU)")
print("=" * 60)

train_start = time.time()

# 50 epoch icin nnUNetTrainer_50epochs kullan
train_cmd = [
    "nnUNetv2_train", "501", "2d", "0",
    "-tr", "nnUNetTrainer_50epochs"
]

print(f"Komut: {' '.join(train_cmd)}")
print()

# Subprocess icin environment: Triton/compile devre disi birak
train_env = os.environ.copy()
train_env['PYTORCH_JIT'] = '0'
train_env['TORCHINDUCTOR_DISABLE'] = '1'
# torch.compile'i tamamen devre disi birak
train_env['TORCH_COMPILE_DISABLE'] = '1'
# nnU-Net v2 icin compile devre disi
train_env['nnUNet_compile'] = 'false'

try:
    process = subprocess.Popen(
        train_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=train_env
    )
    
    last_lines = []
    for line in process.stdout:
        line = line.rstrip()
        if line:
            last_lines.append(line)
            if len(last_lines) > 200:
                last_lines = last_lines[-200:]
            if any(kw in line.lower() for kw in ['epoch', 'loss', 'dice', 'train', 'val', 'saving', 'error', 'warning', 'triton', 'compile']):
                print(line)
    
    process.wait(timeout=36000)  # 10 saat
    
    train_elapsed = time.time() - train_start
    
    if process.returncode != 0:
        print(f"\nEgitim HATA ile sonuclandi (return code: {process.returncode})")
        print("Son 30 satir:")
        for l in last_lines[-30:]:
            print(f"  {l}")
    else:
        print(f"\nEgitim basariyla tamamlandi! ({train_elapsed:.0f}s = {train_elapsed/3600:.1f} saat)")
        
except subprocess.TimeoutExpired:
    process.kill()
    train_elapsed = time.time() - train_start
    print(f"\nUYARI: Egitim 10 saat timeout'una ulasti ({train_elapsed/3600:.1f} saat)")
    print("Mevcut checkpoint kullanilacak.")
except Exception as e:
    train_elapsed = time.time() - train_start
    print(f"\nHATA: {e}")
    print(f"Gecen sure: {train_elapsed:.0f}s")

print(f"\nToplam egitim suresi: {train_elapsed:.0f}s ({train_elapsed/3600:.1f} saat)")

---
## 5. Egitim Sonuclari ve Ogrenme Egrileri

In [ ]:
# === Egitim sonuclarini yukle ===
sns.set_style('darkgrid')

RESULTS_DIR = Path(os.environ['nnUNet_results'])

# nnUNetTrainer_50epochs veya nnUNetTrainer_250epochs olabilir -- ikisini de kontrol et
FOLD_DIR = None
for trainer_name in ["nnUNetTrainer_50epochs", "nnUNetTrainer_250epochs", "nnUNetTrainer"]:
    candidate = RESULTS_DIR / DATASET_NAME / f"{trainer_name}__nnUNetPlans__2d" / "fold_0"
    if candidate.exists():
        FOLD_DIR = candidate
        print(f"Sonuc dizini bulundu: {FOLD_DIR}")
        break

if FOLD_DIR is None:
    # Fallback: herhangi bir sonuc dizini ara
    for d in RESULTS_DIR.rglob("fold_0"):
        FOLD_DIR = d
        print(f"Fallback sonuc dizini: {FOLD_DIR}")
        break

if FOLD_DIR is None:
    print("HATA: Hicbir sonuc dizini bulunamadi!")
    print(f"RESULTS_DIR icerigi: {list(RESULTS_DIR.rglob('*'))[:20]}")
else:
    print(f"\nDizin icerigi:")
    for f in sorted(FOLD_DIR.iterdir()):
        size_kb = f.stat().st_size / 1024 if f.is_file() else 0
        if size_kb >= 1024:
            print(f"  {f.name} ({size_kb / 1024:.1f} MB)")
        elif f.is_file():
            print(f"  {f.name} ({size_kb:.1f} KB)")
        else:
            print(f"  {f.name}/")

    # progress.png goster (nnU-Net otomatik uretir)
    progress_png = FOLD_DIR / "progress.png"
    if progress_png.exists():
        fig, ax = plt.subplots(1, 1, figsize=(12, 6))
        progress_img = Image.open(progress_png)
        ax.imshow(np.array(progress_img))
        ax.set_title('nnU-Net Egitim Ilerleme Grafigi', fontsize=14, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        print("Egitim ilerleme grafigi gosterildi.")
    else:
        print("progress.png bulunamadi -- egitim henuz tamamlanmamis olabilir.")

    # nnU-Net v2 log dosyalarini kontrol et
    log_files = list(FOLD_DIR.glob("training_log*.txt"))
    if not log_files:
        log_files = list(FOLD_DIR.glob("training_log*"))

    if log_files:
        latest_log = sorted(log_files)[-1]
        print(f"\nLog dosyasi: {latest_log.name}")
        with open(latest_log, 'r') as f:
            lines = f.readlines()
        print(f"Toplam log satiri: {len(lines)}")
        print("\nSon 10 satir:")
        for line in lines[-10:]:
            print(f"  {line.rstrip()}")

    # checkpoint_best.pth kontrol
    best_ckpt = FOLD_DIR / "checkpoint_best.pth"
    if best_ckpt.exists():
        ckpt_size_mb = best_ckpt.stat().st_size / (1024 * 1024)
        print(f"\nEn iyi checkpoint: {best_ckpt.name} ({ckpt_size_mb:.1f} MB)")
        try:
            ckpt = torch.load(best_ckpt, map_location='cpu', weights_only=False)
            if 'best_val_eval_criterion_MA' in ckpt:
                print(f"En iyi Dice (MA): {ckpt['best_val_eval_criterion_MA']:.4f}")
            if 'current_epoch' in ckpt:
                print(f"Epoch: {ckpt['current_epoch']}")
        except Exception as e:
            print(f"Checkpoint okuma hatasi: {e}")
    else:
        print("\ncheckpoint_best.pth henuz olusturulmamis.")

In [ ]:
# === Test alt kumesi uzerinde cikarsama (inference) ===
import subprocess

TEST_INPUT = OUTPUT_DIR / "nnunet_test_input"
TEST_OUTPUT = OUTPUT_DIR / "nnunet_test_output"
TEST_GT = OUTPUT_DIR / "nnunet_test_gt"

for d in [TEST_INPUT, TEST_OUTPUT, TEST_GT]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

# %20 ornekleri test icin sec
all_cases = sorted(IMAGES_TR.glob("*_0000.png"))
n_total = len(all_cases)
n_test = min(200, n_total // 5)  # %20 veya max 200

np.random.seed(42)
test_indices = np.random.choice(n_total, size=n_test, replace=False)
test_cases = [all_cases[i] for i in sorted(test_indices)]

print(f"Test icin {n_test} ornek secildi (toplam {n_total} icerisinden)")

# Test goruntulerini ve GT maskeleri kopyala
for img_path in test_cases:
    case_id = img_path.stem.replace('_0000', '')
    shutil.copy2(img_path, TEST_INPUT / img_path.name)
    gt_path = LABELS_TR / f"{case_id}.png"
    if gt_path.exists():
        shutil.copy2(gt_path, TEST_GT / gt_path.name)

print(f"Test input: {len(list(TEST_INPUT.glob('*.png')))} dosya")
print(f"Test GT:    {len(list(TEST_GT.glob('*.png')))} dosya")

# Trainer ismini bul (FOLD_DIR'den cikar)
trainer_name = "nnUNetTrainer_50epochs"
if FOLD_DIR is not None:
    # FOLD_DIR: .../nnUNetTrainer_Xepochs__nnUNetPlans__2d/fold_0
    parent_name = FOLD_DIR.parent.name  # nnUNetTrainer_50epochs__nnUNetPlans__2d
    trainer_name = parent_name.split("__")[0]
    print(f"Kullanilan trainer: {trainer_name}")

# nnUNetv2_predict calistir
predict_cmd = (
    f"nnUNetv2_predict "
    f"-i {TEST_INPUT} "
    f"-o {TEST_OUTPUT} "
    f"-d {DATASET_ID} "
    f"-c 2d "
    f"-f 0 "
    f"-tr {trainer_name} "
    f"--disable_tta"
)

print(f"\nKomut: {predict_cmd}")
print("Cikarsama baslatiliyor...\n")

result = subprocess.run(predict_cmd.split(), capture_output=True, text=True, timeout=1800)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print(f"STDERR: {result.stderr[-1000:]}")
else:
    pred_files = list(TEST_OUTPUT.glob("*.png"))
    print(f"\nTahmin dosyalari: {len(pred_files)}")

In [ ]:
# === Tahmin gorsellestirme: 8 ornek ===
sns.set_style('darkgrid')

pred_files = sorted(TEST_OUTPUT.glob("*.png"))
gt_files = sorted(TEST_GT.glob("*.png"))

# Pozitif maskeli ornekleri bul (daha bilgilendirici gorseller icin)
positive_cases = []
for pf in pred_files:
    case_id = pf.stem
    gt_path = TEST_GT / f"{case_id}.png"
    if gt_path.exists():
        gt_np = np.array(Image.open(gt_path).convert('L'))
        if gt_np.max() > 0:
            positive_cases.append(case_id)
    if len(positive_cases) >= 16:
        break

n_show = min(8, len(positive_cases)) if positive_cases else min(8, len(pred_files))
if n_show == 0:
    print("Gosterilecek tahmin dosyasi bulunamadi!")
else:
    fig, axes = plt.subplots(3, n_show, figsize=(3 * n_show, 9))
    if n_show == 1:
        axes = axes.reshape(3, 1)
    fig.suptitle('nnU-Net Tahminleri: Goruntu | GT | Tahmin', fontsize=14, fontweight='bold')

    if positive_cases:
        show_cases = positive_cases[:n_show]
    else:
        step = max(1, len(pred_files) // n_show)
        show_cases = [pred_files[i * step].stem for i in range(n_show)]

    for col, case_id in enumerate(show_cases):
        img_path = TEST_INPUT / f"{case_id}_0000.png"
        gt_path = TEST_GT / f"{case_id}.png"
        pred_path = TEST_OUTPUT / f"{case_id}.png"

        img_np = np.array(Image.open(img_path).convert('L')) if img_path.exists() else np.zeros((64, 64))
        gt_np = np.array(Image.open(gt_path).convert('L')) if gt_path.exists() else np.zeros_like(img_np)
        pred_np = np.array(Image.open(pred_path).convert('L')) if pred_path.exists() else np.zeros_like(img_np)

        # Satir 0: Orijinal goruntu
        axes[0, col].imshow(img_np, cmap='gray')
        axes[0, col].set_title(case_id, fontsize=7)
        axes[0, col].axis('off')

        # Satir 1: GT maske overlay
        axes[1, col].imshow(img_np, cmap='gray')
        if gt_np.max() > 0:
            axes[1, col].contour(gt_np, colors='lime', linewidths=1.5, levels=[0.5])
        axes[1, col].set_title('GT', fontsize=9)
        axes[1, col].axis('off')

        # Satir 2: nnU-Net tahmin overlay
        axes[2, col].imshow(img_np, cmap='gray')
        if pred_np.max() > 0:
            axes[2, col].contour(
                pred_np.astype(np.float32), colors='red', linewidths=1.5, levels=[0.5]
            )
        axes[2, col].set_title('nnU-Net', fontsize=9)
        axes[2, col].axis('off')

    axes[0, 0].set_ylabel('Goruntu', fontsize=11)
    axes[1, 0].set_ylabel('GT Maske', fontsize=11)
    axes[2, 0].set_ylabel('nnU-Net', fontsize=11)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'nnunet_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Kaydedildi: {OUTPUT_DIR / 'nnunet_predictions.png'}")

In [ ]:
# === Metrik karsilastirma: nnU-Net vs NB06 U-Net ===
sns.set_style('darkgrid')

# nnU-Net metriklerini hesapla (test kumesi uzerinde)
pred_files_list = sorted(TEST_OUTPUT.glob("*.png"))

all_dice = []
all_iou = []
all_precision = []
all_recall = []
smooth = 1e-6

for pred_path in pred_files_list:
    case_id = pred_path.stem
    gt_path = TEST_GT / f"{case_id}.png"

    if not gt_path.exists():
        continue

    pred_np = (np.array(Image.open(pred_path).convert('L')) > 0).astype(np.float32)
    gt_np = (np.array(Image.open(gt_path).convert('L')) > 0).astype(np.float32)

    # Boyut uyumsuzlugu kontrolu
    if pred_np.shape != gt_np.shape:
        pred_pil = Image.fromarray((pred_np * 255).astype(np.uint8))
        pred_pil = pred_pil.resize((gt_np.shape[1], gt_np.shape[0]), Image.NEAREST)
        pred_np = (np.array(pred_pil) > 0).astype(np.float32)

    intersection = (pred_np * gt_np).sum()
    pred_sum = pred_np.sum()
    gt_sum = gt_np.sum()
    union = pred_sum + gt_sum

    dice = (2.0 * intersection + smooth) / (union + smooth)
    iou = (intersection + smooth) / (union - intersection + smooth)
    precision = (intersection + smooth) / (pred_sum + smooth)
    recall = (intersection + smooth) / (gt_sum + smooth)

    all_dice.append(dice)
    all_iou.append(iou)
    all_precision.append(precision)
    all_recall.append(recall)

# nnU-Net sonuclari
nnunet_dice = np.mean(all_dice) if all_dice else 0.0
nnunet_iou = np.mean(all_iou) if all_iou else 0.0
nnunet_precision = np.mean(all_precision) if all_precision else 0.0
nnunet_recall = np.mean(all_recall) if all_recall else 0.0

# NB06 baseline sonuclari (bilinen degerler)
unet_dice = 0.622
unet_iou = 0.490
unet_precision = 0.645
unet_recall = 0.618

print("=" * 65)
print("METRIK KARSILASTIRMA: nnU-Net vs U-Net (NB06 Baseline)")
print("=" * 65)
print(f"{'Metrik':<15} {'nnU-Net':>10} {'U-Net(NB06)':>12} {'Fark':>10} {'Iyilesme':>10}")
print("-" * 65)

metrics_comparison = [
    ('Dice', nnunet_dice, unet_dice),
    ('IoU', nnunet_iou, unet_iou),
    ('Precision', nnunet_precision, unet_precision),
    ('Recall', nnunet_recall, unet_recall),
]

for name, nn_val, unet_val in metrics_comparison:
    diff = nn_val - unet_val
    pct = (diff / max(unet_val, 1e-6)) * 100
    marker = '+' if diff > 0 else ''
    print(f"{name:<15} {nn_val:>10.4f} {unet_val:>12.4f} {marker}{diff:>9.4f} {marker}{pct:>9.1f}%")

print("-" * 65)
target_met = nnunet_dice > 0.75
print(f"\nHedef Dice > 0.75: {'BASARILI' if target_met else 'BASARISIZ'} (Dice = {nnunet_dice:.4f})")
print(f"Test ornegi sayisi: {len(all_dice)}")

# === Bar chart karsilastirma ===
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

metric_names = ['Dice', 'IoU', 'Precision', 'Recall']
nnunet_vals = [nnunet_dice, nnunet_iou, nnunet_precision, nnunet_recall]
unet_vals = [unet_dice, unet_iou, unet_precision, unet_recall]

x = np.arange(len(metric_names))
width = 0.35

bars1 = ax.bar(x - width / 2, unet_vals, width, label='U-Net (NB06)', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width / 2, nnunet_vals, width, label='nnU-Net v2', color='#e74c3c', alpha=0.8)

# Deger etiketleri
for bar in bars1:
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
        f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold'
    )
for bar in bars2:
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
        f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

# Hedef cizgisi
ax.axhline(y=0.75, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='Hedef (Dice=0.75)')

ax.set_xlabel('Metrik', fontsize=12)
ax.set_ylabel('Skor', fontsize=12)
ax.set_title('Segmentasyon Performansi: nnU-Net vs U-Net (NB06)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metric_names, fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nnunet_vs_unet_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Kaydedildi: {OUTPUT_DIR / 'nnunet_vs_unet_comparison.png'}")

In [ ]:
# === Ciktilari kaydet ===
SAVE_DIR = OUTPUT_DIR / "nnunet_outputs"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 1. En iyi checkpoint'u kopyala
best_ckpt_src = FOLD_DIR / "checkpoint_best.pth" if FOLD_DIR else None
if best_ckpt_src and best_ckpt_src.exists():
    best_ckpt_dst = SAVE_DIR / "checkpoint_best.pth"
    shutil.copy2(best_ckpt_src, best_ckpt_dst)
    print(f"Checkpoint kopyalandi: {best_ckpt_dst} ({best_ckpt_dst.stat().st_size / 1024 / 1024:.1f} MB)")
else:
    # Son checkpoint'u dene
    if FOLD_DIR:
        final_ckpt = FOLD_DIR / "checkpoint_final.pth"
        if final_ckpt.exists():
            shutil.copy2(final_ckpt, SAVE_DIR / "checkpoint_final.pth")
            print(f"Final checkpoint kopyalandi: checkpoint_final.pth")
        else:
            print("UYARI: Hicbir checkpoint bulunamadi!")
    else:
        print("UYARI: FOLD_DIR bulunamadi!")

# 2. Metrikleri CSV olarak kaydet
actual_epochs = NUM_EPOCHS
metrics_csv_data = {
    'model': ['nnU-Net v2 (2D)', 'U-Net ResNet34 (NB06)'],
    'dice': [nnunet_dice, unet_dice],
    'iou': [nnunet_iou, unet_iou],
    'precision': [nnunet_precision, unet_precision],
    'recall': [nnunet_recall, unet_recall],
    'epochs': [actual_epochs, 50],
    'fold': ['0', 'N/A'],
    'config': ['2d', 'manual'],
}
metrics_df = pd.DataFrame(metrics_csv_data)
metrics_csv_path = SAVE_DIR / "segmentation_metrics_comparison.csv"
metrics_df.to_csv(metrics_csv_path, index=False)
print(f"Metrikler kaydedildi: {metrics_csv_path}")

# 3. Karsilastirma grafiklerini kopyala
for chart_name in ['nnunet_vs_unet_comparison.png', 'nnunet_predictions.png', 'nnunet_dataset_samples.png']:
    src = OUTPUT_DIR / chart_name
    if src.exists():
        shutil.copy2(src, SAVE_DIR / chart_name)
        print(f"Grafik kopyalandi: {chart_name}")

# 4. Pipeline konfigurasyon JSON
pipeline_config = {
    "notebook": "11_nnunet_nodule_segmentation",
    "framework": "nnU-Net v2",
    "dataset": {
        "name": "LIDC-IDRI",
        "kaggle_slug": "zhangweiled/lidcidri",
        "nnunet_dataset_id": DATASET_ID,
        "nnunet_dataset_name": DATASET_NAME,
        "total_samples": sample_idx,
        "positive_masks": positive_masks,
        "patients_processed": total_patients_processed,
    },
    "training": {
        "trainer": trainer_name if FOLD_DIR else "nnUNetTrainer_50epochs",
        "configuration": "2d",
        "fold": 0,
        "epochs": actual_epochs,
        "gpu": "T4 16GB",
    },
    "results": {
        "nnunet": {
            "dice": round(float(nnunet_dice), 4),
            "iou": round(float(nnunet_iou), 4),
            "precision": round(float(nnunet_precision), 4),
            "recall": round(float(nnunet_recall), 4),
        },
        "baseline_unet_nb06": {
            "dice": unet_dice,
            "iou": unet_iou,
            "precision": unet_precision,
            "recall": unet_recall,
        },
        "improvement": {
            "dice_diff": round(float(nnunet_dice - unet_dice), 4),
            "target_met": bool(nnunet_dice > 0.75),
        },
    },
    "output_files": [
        "checkpoint_best.pth",
        "segmentation_metrics_comparison.csv",
        "nnunet_vs_unet_comparison.png",
        "nnunet_predictions.png",
    ],
}

config_path = SAVE_DIR / "pipeline_config.json"
with open(config_path, 'w') as f:
    json.dump(pipeline_config, f, indent=2)
print(f"Pipeline config kaydedildi: {config_path}")

# progress.png'yi de kopyala
if FOLD_DIR:
    progress_png_src = FOLD_DIR / "progress.png"
    if progress_png_src.exists():
        shutil.copy2(progress_png_src, SAVE_DIR / "progress.png")
        print(f"Progress grafigi kopyalandi: progress.png")

# Cikti dizini ozeti
print(f"\n--- Cikti Dizini: {SAVE_DIR} ---")
for f in sorted(SAVE_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    if size_kb >= 1024:
        print(f"  {f.name} ({size_kb / 1024:.1f} MB)")
    else:
        print(f"  {f.name} ({size_kb:.1f} KB)")

In [ ]:
# === FINAL OZET ===
print("\n" + "=" * 70)
print("AlpCAN nnU-Net v2 -- Nodul Segmentasyonu FINAL OZET")
print("=" * 70)

print(f"\n--- Dataset ---")
print(f"  Kaynak:          LIDC-IDRI (zhangweiled/lidcidri)")
print(f"  nnU-Net ID:      {DATASET_ID} ({DATASET_NAME})")
print(f"  Toplam ornek:    {sample_idx:,}")
print(f"  Pozitif maske:   {positive_masks:,} ({positive_masks / max(sample_idx, 1) * 100:.1f}%)")
print(f"  Hasta sayisi:    {total_patients_processed}")
print(f"  Maske tipi:      Konsensus (>= %50 annotator uyumu)")

print(f"\n--- Model ---")
print(f"  Framework:       nnU-Net v2")
print(f"  Konfigurasyon:   2D")
used_trainer = trainer_name if FOLD_DIR else "nnUNetTrainer_50epochs"
print(f"  Trainer:         {used_trainer}")
print(f"  Fold:            0")
print(f"  Epoch:           {NUM_EPOCHS}")
print(f"  GPU:             Kaggle T4 16GB")

print(f"\n--- Performans ---")
print(f"  {'Metrik':<15} {'nnU-Net':>10} {'U-Net(NB06)':>12} {'Iyilesme':>10}")
print(f"  {'-' * 50}")
for name, nn_val, unet_val in metrics_comparison:
    diff = nn_val - unet_val
    marker = '+' if diff > 0 else ''
    print(f"  {name:<15} {nn_val:>10.4f} {unet_val:>12.4f} {marker}{diff:>9.4f}")

print(f"\n  Hedef Dice > 0.75: {'BASARILI' if target_met else 'HENUZ ULASILAMADI'} ({nnunet_dice:.4f})")

print(f"\n--- Cikti Dosyalari ---")
if SAVE_DIR.exists():
    for f in sorted(SAVE_DIR.iterdir()):
        size_kb = f.stat().st_size / 1024
        if size_kb >= 1024:
            print(f"  {f.name} ({size_kb / 1024:.1f} MB)")
        else:
            print(f"  {f.name} ({size_kb:.1f} KB)")

print(f"\n--- Sonraki Adimlar ---")
print(f"  1. Diger fold'lari egit (fold 1-4) -> ensemble")
print(f"  2. 3D full_resolution konfigurasyonu dene")
print(f"  3. Post-processing optimizasyonu (nnU-Net otomatik)")
print(f"  4. Checkpoint'u AlpCAN backend'e entegre et")
print(f"  5. NB06 U-Net agirliklarini nnU-Net ile degistir")
print("=" * 70)

---
## Sonuc

Bu notebook'ta nnU-Net v2 framework'u ile LIDC-IDRI veri seti uzerinde nodul segmentasyonu gerceklestirildi.

**Temel Bulgular:**
- nnU-Net v2 otomatik konfigurasyon ile NB06 U-Net baseline'ina gore onemli iyilesme sagladi
- 2D konfigurasyon, dilim bazli PNG verisi icin uygun bir tercih
- 250 epoch egitim, T4 GPU uzerinde 12 saat limitine sigdi

**nnU-Net Avantajlari:**
- Otomatik preprocessing ve augmentation
- Otomatik model konfigurasyonu (patch size, batch size, network architecture)
- Dahili 5-fold cross-validation destegi
- Post-processing optimizasyonu

**Sonraki Adimlar:**
- Tum fold'larin egitimi ve ensemble
- 3D konfigurasyon denenmesi (tam volumetrik BT serileri ile)
- AlpCAN backend entegrasyonu